In [7]:
import numpy as np
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.optimize import minimize
from pymoo.indicators.gd import GD
from pymoo.indicators.igd import IGD
from pymoo.indicators.hv import HV

# SCH Problem
class SCH(Problem):
    def __init__(self):
        super().__init__(n_var=1, n_obj=2, xl=[-1000.0], xu=[1000.0])

    def _evaluate(self, x, out, *args, **kwargs):
        out["F"] = np.column_stack([x[:, 0]**2, (x[:, 0] - 2)**2])

# Run NSGA-II
res = minimize(SCH(), NSGA2(pop_size=50), ('n_evals', 800), seed=42, verbose=False)

# True Pareto Front
x_true = np.linspace(0, 2, 100)
pf_true = np.column_stack([x_true**2, (x_true - 2)**2])

# Quality Indicators
if res.F is not None:
    print("--- Optimization Results ---")
    print(f"GD:  {GD(pf_true)(res.F):.4f}")
    print(f"IGD: {IGD(pf_true)(res.F):.4f}")
    print(f"HV:  {HV(ref_point=np.array([5.0, 5.0]))(res.F):.4f}")
    print(f"Pareto Points: {len(res.F)}/50 ({len(res.F) * 2}%)")

    # Spread (Delta) — guarded against single-point fronts
    F = res.F[np.argsort(res.F[:, 0])]
    if len(F) < 2:
        print("Spread (Delta): N/A (insufficient Pareto points)")
    else:
        d_i = np.linalg.norm(np.diff(F, axis=0), axis=1)
        d_bar = d_i.mean()
        df = np.linalg.norm(pf_true[0]  - F[0])
        dl = np.linalg.norm(pf_true[-1] - F[-1])
        denom = df + dl + (len(F) - 1) * d_bar
        spread = (df + dl + np.sum(np.abs(d_i - d_bar))) / denom if denom > 0 else float('nan')
        print(f"Spread (Delta): {spread:.4f}")
else:
    print("No feasible solutions found.")

--- Optimization Results ---
GD:  0.0425
IGD: 0.3704
HV:  20.7317
Pareto Points: 14/50 (28%)
Spread (Delta): 1.0853
